In [2]:
pip install pandas

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --------------- ------------------------ 3.7/9.7 MB 19.8 MB/s eta 0:00:01
   ------------------- -------------------- 4.7/9.7 MB 13.0 MB/s eta 0:00:01
   ----------------------- ---------------- 5.8/9.7 MB 9.5 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/9.7 MB 7.9 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.7 MB 6.6 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.7 MB 6.2 MB/s eta 0:00:01
   ----------------------------------- ---- 8.7/9.7 MB 5.8 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 5.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.3 MB 6.3 MB/s eta 0:00:02
   ------ --------------------------------- 2.1/12.3 MB 5.1 MB/s eta 0:00:03
   -------- ------------------------------- 2.6/12.3 MB 4.6 MB/s eta 0:00:03
   ---------- ------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

areas = pd.read_csv("areas.csv")
price = pd.read_csv("price_data.csv")
congestion = pd.read_csv("congestion_data.csv")
aqi = pd.read_csv("aqi_data.csv")
growth = pd.read_csv("growth_score.csv")

In [4]:
df = areas.merge(price, on="area_id") \
          .merge(congestion, on="area_id") \
          .merge(aqi, on="area_id") \
          .merge(growth, on="area_id")

In [12]:
print(df.head())

   area_id    area_name  price_per_sqft  congestion_score  aqi  growth_score
0        1        Wakad            9000                 7  155             8
1        2    Hinjawadi            8800                 9  185            10
2        3  Viman Nagar           12800                 8  175             8
3        4      Kharadi           10500                 7  190             9
4        5        Aundh           12000                 6  150             7


In [6]:
def normalize(col):
    return (col - col.min()) / (col.max() - col.min())

In [7]:
# Normalization
df["price_norm"] = normalize(df["price_per_sqft"])
df["congestion_norm"] = normalize(df["congestion_score"])
df["aqi_norm"] = normalize(df["aqi"])
df["growth_norm"] = normalize(df["growth_score"])

In [8]:
# Invert bad factors
df["price_score"] = 1 - df["price_norm"]
df["congestion_score_final"] = 1 - df["congestion_norm"]
df["aqi_score"] = 1 - df["aqi_norm"]

In [9]:
# Final Livability Score
df["final_score"] = (
    0.3 * df["price_score"] +
    0.25 * df["congestion_score_final"] +
    0.2 * df["aqi_score"] +
    0.25 * df["growth_norm"]
)

In [10]:
# Rank Areas
df["rank"] = df["final_score"].rank(ascending=False)
df = df.sort_values(by="final_score", ascending=False)

In [11]:
df.to_csv("final_livability_data.csv", index=False)

In [16]:
df_clean = df[[
    "area_id",
    "area_name",
    "price_per_sqft",
    "congestion_score",
    "aqi",
    "growth_score",
    "final_score",
    "rank"
]]

df_clean.to_csv("final_clean.csv", index=False)

In [19]:
print(df_clean.head())

   area_id        area_name  price_per_sqft  congestion_score  aqi  \
9       10            Ravet            7000                 4  135   
0        1            Wakad            9000                 7  155   
7        8         Balewadi           10900                 5  140   
1        2        Hinjawadi            8800                 9  185   
6        7  Pimple Saudagar            9500                 6  145   

   growth_score  final_score  rank  
9             7     0.812500   1.0  
0             8     0.554885   2.0  
7             7     0.544109   3.0  
1            10     0.490230   4.0  
6             6     0.487356   5.0  
